# Tutorial 3: Circuit QED Parameters — Josephson Junctions, Transmons, and Unit Conversions

<table>
    <tr>
        <td>Author:</td>
        <td>Zlatko Minev</td>
    </tr>
    <tr>
        <td>Purpose:</td>
        <td>Demonstrate the circuit QED conversion utilities in <code>pyEPR.calcs</code>: unit conversions, Josephson junction parameters, transmon physics, and physical constants.</td>
    </tr>
    <tr>
        <td>File Status:</td>
        <td>Complete</td>
    </tr>
</table>

**References:**
- Koch et al., *Charge-insensitive qubit design derived from the Cooper pair box*, PRA **76**, 042319 (2007) — the foundational transmon paper.
- Minev et al., *Energy-participation quantization of Josephson circuits*, npj Quantum Information (2021); arXiv:2010.00620 — the EPR framework implemented in pyEPR.
- Devoret & Schoelkopf, *Superconducting Circuits for Quantum Information: An Outlook*, Science **339**, 1169 (2013) — broad review of circuit QED.

## Introduction: The Key Players in Superconducting Qubit Design

Superconducting qubits are built from two ingredients: **linear electromagnetic resonators** (modelled as LC oscillators) and **Josephson junctions** (JJs), which provide the essential nonlinearity. Without a nonlinear element, a purely harmonic oscillator has equally spaced energy levels and cannot be selectively addressed as a two-level system.

### The Josephson Junction

A Josephson junction is a thin insulating barrier sandwiched between two superconducting electrodes. Its constitutive relation is:

$$I = I_c \sin\varphi, \qquad V = \frac{\Phi_0}{2\pi}\dot{\varphi},$$

where $\varphi$ is the gauge-invariant phase difference across the junction, $I_c$ is the critical current, and $\Phi_0 = h/(2e)$ is the (superconducting) flux quantum. These two equations together define a purely inductive, *nonlinear* element.

The potential energy stored in the junction is:

$$U_J(\varphi) = -E_J \cos\varphi,$$

where the **Josephson energy** $E_J$ sets the depth of the cosine potential well:

$$E_J = \frac{\Phi_0}{2\pi} I_c = \frac{\hbar}{2e} I_c = \frac{\phi_0^2}{L_J}.$$

Here $\phi_0 = \Phi_0/(2\pi) = \hbar/(2e)$ is the *reduced* flux quantum.

### Linearised Josephson Inductance

For small oscillations around $\varphi = 0$, the cosine potential is well approximated as a parabola:

$$U_J \approx \frac{1}{2} E_J \varphi^2 + \mathcal{O}(\varphi^4).$$

This means the junction looks like a linear inductor with inductance:

$$L_J = \frac{\phi_0^2}{E_J} = \frac{(\Phi_0/2\pi)^2}{E_J}.$$

This **linearised Josephson inductance** $L_J$ is the quantity that enters the electromagnetic simulation in HFSS. The EPR method extracts the zero-point fluctuations of the reduced flux $\varphi_{\text{zpf}}$ across the junction, which then feeds into the full nonlinear analysis.

### The Charging Energy

Each electrode of the junction has a capacitance $C_\Sigma$ (including the junction self-capacitance and any shunting capacitors). The **charging energy** is:

$$E_C = \frac{e^2}{2 C_\Sigma},$$

which is the energy cost of placing one Cooper pair ($2e$) on the capacitor. Note that some references define $E_C = e^2/(2C)$ while others write $4E_C$ for the charging energy — the convention used here follows Koch et al. (2007), where $E_C = e^2/(2C_\Sigma)$.

### The Transmon Hamiltonian

A transmon qubit is a Cooper-pair box shunted with a large capacitor to increase $C_\Sigma$ and suppress charge noise. The Hamiltonian (in the charge basis, ignoring offset charge) is:

$$\hat{H} = 4 E_C \hat{n}^2 - E_J \cos\hat{\varphi},$$

where $\hat{n}$ is the Cooper-pair number operator conjugate to $\hat{\varphi}$. The key dimensionless ratio is:

$$\frac{E_J}{E_C} \gg 1 \quad \text{(transmon regime)}.$$

Typically, $E_J/E_C \sim 50$–$100$ for a transmon. In this limit:

- The qubit transition frequency approaches the plasma frequency: $f_{01} \approx \sqrt{8 E_J E_C}/h$.
- The **anharmonicity** (the difference between the $|1\rangle\to|2\rangle$ and $|0\rangle\to|1\rangle$ transition frequencies) is approximately $\alpha \approx -E_C$, which is small but sufficient to address the qubit selectively.
- The qubit is exponentially insensitive to charge noise for $E_J/E_C \gtrsim 50$.

### Connection to EPR

The EPR method (arXiv:2010.00620) links the HFSS eigenmode simulation directly to the quantum Hamiltonian. The zero-point flux fluctuation across junction $j$ in mode $m$ is:

$$\varphi_{\text{zpf}}^{(j,m)} = \sqrt{\frac{\hbar}{2} \cdot \frac{\omega_m}{E_J^{(j)}}},$$

or equivalently $\varphi_{\text{zpf}} = (2E_C/E_J)^{1/4}$ in the transmon limit. The participation ratio $p_{jm}$ is the fraction of the mode's inductive energy stored in junction $j$. Together these give the anharmonicity and cross-Kerr couplings directly from the HFSS solution.

The utilities in this tutorial let you convert between all the representations ($L_J$, $I_c$, $E_J$, $E_C$, $C$) and compute the transmon spectrum, complementing the HFSS-based EPR workflow.

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Unit Conversions

### Why unit conversions matter

pyEPR works internally in SI units (metres, henries, farads, joules, radians/second), but circuit QED practitioners invariably think in convenient scaled units:

| Quantity | Practical unit | SI equivalent |
|----------|---------------|---------------|
| Inductance | nH (nano-henry) | $10^{-9}$ H |
| Capacitance | fF (femto-farad) | $10^{-15}$ F |
| Frequency / Energy | GHz | $h \times 10^9$ J |
| Frequency / Energy | MHz | $h \times 10^6$ J |
| Current | nA (nano-ampere) | $10^{-9}$ A |

The `Convert` class in `pyEPR.calcs` provides bidirectional conversion between SI and these practical units, as well as conversions between physically equivalent representations (e.g., $L_J \leftrightarrow E_J \leftrightarrow I_c$, and $C_\Sigma \leftrightarrow E_C$).

Note: These conversions are intentionally kept as a lightweight internal implementation — no external dependency is introduced for what are essentially a handful of scaling factors and physical constants.

### Elementary SI conversions

`Convert.toSI(value, unit)` scales a value *into* SI (multiply by the unit's SI prefix factor). `Convert.fromSI(value, unit)` scales a value *out of* SI (divide by the prefix). The round-trip (identity) test confirms correctness.

In [ ]:
import pyEPR.calcs
from pyEPR.calcs import Convert
print("Convert.toSI(1,'nH')     = ", Convert.toSI(1,'nH'), "H")
print("Convert.fromSI(1.0,'nH') = ", Convert.fromSI(1.0,'nH'), "nH")
print("Identity: ",                  Convert.toSI(Convert.fromSI(1.0,'nH'),'nH'))

## 2. Josephson Junction Parameters: $E_J$, $L_J$, $E_C$, and $C_\Sigma$

### Josephson energy from inductance: $E_J \leftrightarrow L_J$

The Josephson energy and the linearised Josephson inductance are related by:

$$E_J = \frac{\phi_0^2}{L_J} = \frac{(\Phi_0/2\pi)^2}{L_J} = \frac{\hbar^2}{4e^2 L_J},$$

or in practical units:

$$E_J \, [\text{GHz}] = \frac{\phi_0^2}{L_J \, [\text{H}] \cdot h \, [\text{J}\cdot\text{s}]} \times 10^{-9}.$$

For a typical transmon junction with $L_J = 10\,\text{nH}$:

$$E_J \approx \frac{(3.29 \times 10^{-16}\,\text{Wb})^2}{10 \times 10^{-9}\,\text{H} \times 6.626 \times 10^{-34}\,\text{J}\cdot\text{s}} \approx 16.35\,\text{GHz},$$

which is a canonical value you will see throughout the transmon literature.

### Charging energy from capacitance: $E_C \leftrightarrow C_\Sigma$

The charging energy follows from:

$$E_C = \frac{e^2}{2 C_\Sigma},$$

or in practical units:

$$E_C \, [\text{MHz}] = \frac{(1.602 \times 10^{-19}\,\text{C})^2}{2 \times C_\Sigma \, [\text{F}] \times h \, [\text{J}\cdot\text{s}]} \times 10^{-6}.$$

For a typical transmon shunting capacitance $C_\Sigma = 65\,\text{fF}$, this gives $E_C \approx 300\,\text{MHz}$, so $E_J/E_C \approx 54$ — solidly in the transmon regime.

The code below exercises both conversions and their inverses.

In [ ]:
from IPython.display import Latex
Lj = 10
display(Latex(r"$E_J = %.2f \text{ GHz} \qquad \text{for } L_J=%.2f\text{ nH}$" % (\
        Convert.Ej_from_Lj(Lj, 'nH', "GHz"),Lj)))

print('\nConvert back %.2f nH' % Convert.Lj_from_Ej(16.35E3, 'MHz', 'nH'),'\n')

display(Latex(r"$E_C = %.2f \text{ MHz} \qquad \text{for } C_J=%.2f\text{ fF}$" % (\
        Convert.Ec_from_Cs(65., 'fF', "MHz"),65.)))

display( 'Convert back:',Latex(r"$C_J = %.2f \text{ fF} \qquad \text{for } E_C=%.2f\text{ MHz}$" % (\
        Convert.Cs_from_Ec(300, 'MHz', "fF"),300)))

## 3. Critical Current: $I_c \leftrightarrow L_J$

The critical current $I_c$ is the maximum supercurrent the junction can carry. It is related to the linearised inductance by:

$$I_c = \frac{\phi_0}{L_J} = \frac{E_J}{\phi_0},$$

where $\phi_0 = \Phi_0/(2\pi) \approx 3.29 \times 10^{-16}\,\text{Wb}$ is the reduced flux quantum. Numerically, for $L_J = 10\,\text{nH}$:

$$I_c = \frac{3.29 \times 10^{-16}}{10 \times 10^{-9}} \approx 32.9\,\text{nA}.$$

This is consistent with the Ambegaokar–Baratoff relation at $T=0$:

$$I_c = \frac{\pi \Delta}{2 e R_N},$$

where $\Delta$ is the superconducting gap and $R_N$ is the junction's normal-state resistance. For aluminium ($\Delta \approx 170\,\mu\text{eV}$), a critical current of $30\,\text{nA}$ corresponds to a normal resistance $R_N \approx 9\,\text{k}\Omega$, which is a typical fabrication target for transmon junctions.

**Key point:** $I_c$ is directly measurable and is the primary specification used in junction fabrication. The pyEPR conversion functions let you move fluently between $I_c$ (a fabrication parameter) and $L_J$ (the simulation parameter).

In [ ]:
print(Convert.Ic_from_Lj(10))
Convert.Lj_from_Ic(32)

## 4. Physical Constants

All the conversion formulas above involve a small set of fundamental constants. pyEPR exposes these directly from `pyEPR.calcs.convert` so that you can write raw physical calculations without relying on external libraries.

| Symbol | Name | Value |
|--------|------|-------|
| $h$ (`Planck`) | Planck constant | $6.626 \times 10^{-34}$ J$\cdot$s |
| $\hbar$ (`hbar`, `ħ`) | Reduced Planck constant | $h/(2\pi) = 1.055 \times 10^{-34}$ J$\cdot$s |
| $e$ (`e_el`, `elementary_charge`) | Elementary charge | $1.602 \times 10^{-19}$ C |
| $\Phi_0$ (`fluxQ`) | Flux quantum ($h/2e$) | $2.068 \times 10^{-15}$ Wb |
| $\phi_0$ (`ϕ0`) | Reduced flux quantum ($\Phi_0/2\pi = \hbar/2e$) | $3.291 \times 10^{-16}$ Wb |
| $\pi$ (`π`, `pi`) | Pi | $3.14159\ldots$ |

The reduced flux quantum $\phi_0 = \hbar/(2e)$ is particularly important: it is the natural unit for the phase variable $\varphi$, and it determines both the Josephson inductance ($L_J = \phi_0^2/E_J$) and the flux quantisation condition.

The cell below performs a raw calculation of $E_J$ using these constants, as a cross-check of the formula and the `Convert` functions.

In [ ]:
from pyEPR.calcs.convert import π, pi, ϕ0, fluxQ, Planck, ħ, hbar, elementary_charge, e_el
print("Test EJ raw calculation = %.2f"%( ϕ0**2 / (10E-9 * Planck) *1E-9 ) ,'GHz')

The result $E_J \approx 16.35\,\text{GHz}$ for $L_J = 10\,\text{nH}$ is confirmed, matching the `Convert.Ej_from_Lj` result above. This is the fundamental cross-check: when you type `ϕ0**2 / (10E-9 * Planck)` you are directly evaluating the defining formula $E_J = \phi_0^2 / L_J$, expressed in joules, and then converting to GHz by dividing by $h$ and scaling by $10^{-9}$.

## 5. Transmon Qubit Physics

### From nonlinear oscillator to weakly anharmonic qubit

The transmon Hamiltonian $\hat{H} = 4E_C \hat{n}^2 - E_J\cos\hat{\varphi}$ can be analysed in the phase basis. In the transmon limit $E_J/E_C \gg 1$, the junction phase is well localised near $\varphi = 0$, and the cosine can be expanded:

$$\hat{H} \approx 4E_C \hat{n}^2 + \frac{1}{2}E_J\hat{\varphi}^2 - \frac{1}{24}E_J\hat{\varphi}^4 + \ldots$$

The first two terms are a harmonic oscillator (the **plasma** oscillator) with angular frequency:

$$\omega_p = \sqrt{8 E_J E_C} / \hbar, \quad \text{or} \quad f_p = \frac{\sqrt{8 E_J E_C}}{h}.$$

The third term ($-E_J\hat{\varphi}^4/24$) is the leading nonlinear correction. In second-order perturbation theory it shifts the $|1\rangle \to |2\rangle$ transition relative to $|0\rangle \to |1\rangle$ by:

$$\alpha \equiv f_{12} - f_{01} \approx -E_C / h,$$

where $f_{01}$ is the qubit frequency and $\alpha$ is the **anharmonicity**. For a typical transmon with $E_C/h \approx 300\,\text{MHz}$, the anharmonicity is about $-300\,\text{MHz}$, which is sufficient to selectively drive the $|0\rangle \leftrightarrow |1\rangle$ transition with nanosecond pulses.

### Zero-point fluctuations

Treating the transmon as a quantum harmonic oscillator, the zero-point fluctuation of the reduced flux $\hat{\varphi}$ is:

$$\varphi_{\text{zpf}} = \left(\frac{2E_C}{E_J}\right)^{1/4}.$$

This quantity is central to the EPR analysis: it sets the strength of the nonlinearity. In the transmon limit, $\varphi_{\text{zpf}} \ll 1$ (small oscillations), which is consistent with the perturbative treatment. The full EPR formula for the Kerr nonlinearity is $\chi \approx -E_C (\varphi_{\text{zpf}})^4 / 2$ per mode.

### Charge insensitivity

The exponential suppression of charge noise in the transmon regime is one of its most important features. The charge dispersion (sensitivity of $f_{01}$ to offset charge) decays exponentially as:

$$\varepsilon_m \propto e^{-\sqrt{8 E_J/E_C}}.$$

For $E_J/E_C = 50$, this suppression is enormous ($\sim e^{-20}$), making the transmon essentially charge-insensitive even though it is still formally a charge qubit.

### The `transmon_print_all_params` function

`CalcsTransmon.transmon_print_all_params(Lj_nH, Cs_fF)` is a convenience function that takes the junction inductance (in nH) and shunting capacitance (in fF), computes $E_J$ and $E_C$, and prints the full set of transmon parameters in a formatted display. This is useful for a quick sanity check during device design.

For the canonical transmon parameters $L_J = 13\,\text{nH}$, $C_\Sigma = 65\,\text{fF}$:

In [ ]:
pyEPR.calcs.CalcsTransmon.transmon_print_all_params(13, 65);

The displayed quantities include:

- **$E_J$, $E_C$** (in MHz and GHz): the fundamental circuit parameters.
- **$f_{01}$** (in GHz): the qubit transition frequency $\approx \sqrt{8 E_J E_C}/h$.
- **$\alpha$** (anharmonicity, in MHz): $\approx -E_C/h$.
- **$\varphi_{\text{zpf}}$** (dimensionless): zero-point fluctuation of the reduced phase, $(2E_C/E_J)^{1/4}$.
- **$Z$** (in $\Omega$): characteristic impedance of the linearised junction-capacitor circuit, $Z = \sqrt{L_J/C_\Sigma}$.

### Raw dictionary output

The underlying function `transmon_get_all_params(Ej_MHz, Ec_MHz)` returns the same quantities as a Python dictionary, which is convenient for programmatic access and post-processing.

In [ ]:
pyEPR.calcs.CalcsTransmon.transmon_get_all_params(Convert.Ej_from_Lj(13, 'nH', 'MHz'), Convert.Ec_from_Cs(65, 'fF', 'MHz'))

Some key values from the output to understand:

- `Phi_ZPF` (in Wb): the dimensional zero-point flux fluctuation $\Phi_{\text{zpf}} = \phi_0 \cdot \varphi_{\text{zpf}}$.
- `Q_ZPF` (in C): the zero-point charge fluctuation $Q_{\text{zpf}} = \hbar / (2\Phi_{\text{zpf}})$.
- `phi_ZPF` (dimensionless): the reduced flux ZPF $\varphi_{\text{zpf}} = (2E_C/E_J)^{1/4}$.
- `n_ZPF`: the Cooper-pair number ZPF $n_{\text{zpf}} = 1/(4\varphi_{\text{zpf}})$.
- `Omega_MHz`: the angular plasma frequency $\omega_p/(2\pi)$ in MHz... wait — this is actually $\omega_p$ in rad/s divided by $2\pi$ and converted, i.e. the plasma frequency $f_p = \sqrt{8E_JE_C}/h$.
- `f_MHz`: the qubit frequency $f_{01} \approx f_p$ in GHz (note units in key name are legacy; value is in GHz).
- `Z_Ohms`: the impedance $Z = \sqrt{L_J/C_\Sigma}$ in ohms, which determines the magnitude of vacuum fluctuations and hence coupling strengths to microwave cavities.

For the parameters here ($L_J = 13\,\text{nH}$, $C_\Sigma = 65\,\text{fF}$): $E_J/E_C \approx 12574/298 \approx 42$, which is in the transmon regime but on the lower end. Increasing $C_\Sigma$ (adding more shunting capacitance) would push $E_J/E_C$ higher, improving charge insensitivity at the cost of reduced anharmonicity.

## Summary: Connecting Circuit Parameters to the EPR Workflow

The conversion utilities in this tutorial sit at the interface between fabrication/design and simulation:

```
Fabrication target
      I_c  ──────────────────────────────► L_J  (simulation input)
                                             │
                                             ▼
                                    HFSS eigenmode simulation
                                             │
                                     EPR analysis extracts
                                      φ_zpf per mode/junction
                                             │
                                             ▼
                                    Quantum Hamiltonian
                                    ω_01, α, χ (cross-Kerr)
```

The workflow is:

1. **Choose** $E_J$ and $E_C$ to hit a target qubit frequency and anharmonicity, using `transmon_print_all_params`.
2. **Convert** $E_J \to L_J$ with `Convert.Lj_from_Ej` to get the HFSS junction parameter.
3. **Convert** $E_J \to I_c$ with `Convert.Ic_from_Lj` to specify the junction to your fabrication team.
4. **Run HFSS** with the linearised inductance $L_J$, then use `DistributedAnalysis` (Tutorial 1) to extract $p_{jm}$ and $\varphi_{\text{zpf}}$.
5. **Compute** the full Hamiltonian with `QuantumAnalysis` (Tutorial 1), recovering $\alpha$, $\chi$, etc.

For more details, see the EPR paper (arXiv:2010.00620) and Koch et al., PRA **76**, 042319 (2007).